In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Core Functions

This module provides a fast.ai-style Pythonic wrapper around Pulumi for Infrastructure as Code provisioning with production-grade security defaults.

In [ ]:
#| export
import os
import pulumi
from pulumi import automation as auto
from typing import Optional, Dict, Any, Callable
from pathlib import Path

### Stack Management

Easy stack creation and management without CLI interaction.

In [ ]:
#| export
class PulumiStack:
    """A fast.ai-style wrapper for Pulumi stacks with sensible defaults and production-grade security."""
    
    def __init__(self, 
                 stack_name: str,
                 project_name: str = "pullup-project",
                 work_dir: Optional[str] = None,
                 backend_url: Optional[str] = None,
                 passphrase: Optional[str] = None):
        """
        Initialize a Pulumi stack with automatic configuration.
        
        Args:
            stack_name: Name of the stack (e.g., 'dev', 'prod')
            project_name: Name of the Pulumi project
            work_dir: Working directory for Pulumi files (defaults to current directory)
            backend_url: Pulumi backend URL (defaults to local file backend)
            passphrase: Passphrase for secrets encryption (reads from PULUMI_CONFIG_PASSPHRASE env var if not provided)
        """
        self.stack_name = stack_name
        self.project_name = project_name
        self.work_dir = work_dir or os.getcwd()
        self.backend_url = backend_url or f"file://{os.path.join(self.work_dir, '.pulumi')}"
        # Use provided passphrase, or environment variable, or generate a default
        self.passphrase = passphrase or os.environ.get('PULUMI_CONFIG_PASSPHRASE', 'pullup-default-passphrase')
        self.stack = None
        self._program = None
        
    def setup(self, program: Callable[[], None]):
        """Setup the stack with a Pulumi program."""
        self._program = program
        return self
    
    def login(self):
        """Login to Pulumi backend (no CLI interaction required)."""
        # Create backend directory if it doesn't exist (for file backend)
        if self.backend_url.startswith('file://'):
            backend_path = self.backend_url.replace('file://', '').split('?')[0]
            Path(backend_path).mkdir(parents=True, exist_ok=True)
        
        # Set environment variables for Pulumi
        os.environ["PULUMI_BACKEND_URL"] = self.backend_url
        os.environ["PULUMI_CONFIG_PASSPHRASE"] = self.passphrase
        return self
    
    def create_or_select(self):
        """Create or select the stack."""
        if self._program is None:
            raise ValueError("Program not set. Call setup() first.")
        
        # Create stack or select if it exists
        self.stack = auto.create_or_select_stack(
            stack_name=self.stack_name,
            project_name=self.project_name,
            program=self._program,
            opts=auto.LocalWorkspaceOptions(
                work_dir=self.work_dir,
                env_vars={
                    "PULUMI_BACKEND_URL": self.backend_url,
                    "PULUMI_CONFIG_PASSPHRASE": self.passphrase
                }
            )
        )
        return self
    
    def set_config(self, key: str, value: str, secret: bool = False):
        """Set configuration value."""
        if self.stack is None:
            raise ValueError("Stack not created. Call create_or_select() first.")
        self.stack.set_config(key, auto.ConfigValue(value=value, secret=secret))
        return self
    
    def up(self, on_output: Optional[Callable] = None):
        """Deploy the stack."""
        if self.stack is None:
            raise ValueError("Stack not created. Call create_or_select() first.")
        return self.stack.up(on_output=on_output)
    
    def preview(self):
        """Preview changes without deploying."""
        if self.stack is None:
            raise ValueError("Stack not created. Call create_or_select() first.")
        return self.stack.preview()
    
    def destroy(self, on_output: Optional[Callable] = None):
        """Destroy all resources in the stack."""
        if self.stack is None:
            raise ValueError("Stack not created. Call create_or_select() first.")
        return self.stack.destroy(on_output=on_output)
    
    def get_outputs(self) -> Dict[str, Any]:
        """Get stack outputs."""
        if self.stack is None:
            raise ValueError("Stack not created. Call create_or_select() first.")
        return self.stack.outputs()
    
    def refresh(self):
        """Refresh the stack state."""
        if self.stack is None:
            raise ValueError("Stack not created. Call create_or_select() first.")
        return self.stack.refresh()

### Helper Functions

Convenience functions for common operations.

In [ ]:
#| export
def quick_stack(stack_name: str, 
                program: Callable[[], None],
                project_name: str = "pullup-project",
                config: Optional[Dict[str, Any]] = None,
                auto_deploy: bool = False,
                secret_keywords: Optional[list] = None) -> PulumiStack:
    """
    Quickly create and configure a Pulumi stack with sensible defaults.
    
    Args:
        stack_name: Name of the stack
        program: Pulumi program function
        project_name: Name of the project
        config: Configuration dictionary
        auto_deploy: If True, automatically deploy the stack
        secret_keywords: List of keywords to identify secret config keys (defaults to common secret patterns)
    
    Returns:
        Configured PulumiStack instance
    """
    # Default secret keywords if not provided
    if secret_keywords is None:
        secret_keywords = [
            'password', 'passwd', 'pwd',
            'token', 'secret', 'apikey', 'api_key',
            'key', 'credential', 'auth',
            'private', 'access_key', 'secret_key'
        ]
    
    stack = PulumiStack(stack_name, project_name)
    stack.login().setup(program).create_or_select()
    
    if config:
        for key, value in config.items():
            # Detect if value should be secret based on key name
            key_lower = key.lower().replace('-', '_')
            is_secret = any(keyword in key_lower for keyword in secret_keywords)
            stack.set_config(key, str(value), secret=is_secret)
    
    if auto_deploy:
        stack.up()
    
    return stack

In [ ]:
#| export
def create_secure_tags(environment: str = "dev", 
                      owner: str = "pullup",
                      **kwargs) -> Dict[str, str]:
    """
    Create a standard set of security and compliance tags.
    
    Args:
        environment: Environment name (dev, staging, prod)
        owner: Owner/team name
        **kwargs: Additional custom tags
    
    Returns:
        Dictionary of tags
    """
    tags = {
        "Environment": environment,
        "ManagedBy": "pullup",
        "Owner": owner,
        "CreatedBy": "pulumi-automation",
    }
    tags.update(kwargs)
    return tags

### Security Defaults

Helper functions for production-grade security configurations.

In [ ]:
#| export
class SecurityDefaults:
    """Production-grade security defaults for cloud resources."""
    
    @staticmethod
    def storage_encryption() -> Dict[str, Any]:
        """Default encryption settings for storage."""
        return {
            "encryption_enabled": True,
            "encryption_type": "AES256",
            "versioning_enabled": True,
        }
    
    @staticmethod
    def network_security() -> Dict[str, Any]:
        """Default network security settings."""
        return {
            "enable_network_policies": True,
            "enable_private_endpoint": True,
            "public_access_enabled": False,
        }
    
    @staticmethod
    def compute_security() -> Dict[str, Any]:
        """Default compute security settings."""
        return {
            "enable_monitoring": True,
            "enable_auto_updates": True,
            "disable_password_auth": True,
            "enable_secure_boot": True,
        }
    
    @staticmethod
    def all_defaults() -> Dict[str, Dict[str, Any]]:
        """Get all security defaults."""
        return {
            "storage": SecurityDefaults.storage_encryption(),
            "network": SecurityDefaults.network_security(),
            "compute": SecurityDefaults.compute_security(),
        }

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()